## 04 - Modélisation multiclasse: CV, tuning & comparaison
Modèles: LogisticRegression, RandomForest, HistGradientBoosting
Métriques: Macro F1 + focus recall de "<30"

In [5]:
from sklearn.metrics import precision_recall_fscore_support
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

DATA_PATH = os.path.join("..", "data", "processed", "dataset_model.csv")
df = pd.read_csv(DATA_PATH)

TARGET = "readmitted"
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str)

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("X shape:", X.shape, "Num:", len(num_cols), "Cat:", len(cat_cols))
print("Target:", y.value_counts())

C:\Users\HL\AppData\Local\Temp\ipykernel_2020\1730346924.py:17: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


X shape: (101766, 52) Num: 13 Cat: 39
Target: readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


In [6]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

pre = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average="macro")

## 1) Logistic Regression (multinomial)

In [7]:
pipe_log = Pipeline(steps=[
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=3000))
])

param_log = {
    "clf__C": np.logspace(-2, 1, 10),
    "clf__solver": ["lbfgs"]
}

search_log = RandomizedSearchCV(
    pipe_log, param_distributions=param_log, n_iter=10,
    scoring=scorer, cv=cv, n_jobs=-1, random_state=42, verbose=1
)
search_log.fit(X_train, y_train)
best_log = search_log.best_estimator_
pred_log = best_log.predict(X_test)

print("LOG best:", search_log.best_params_)
print(classification_report(y_test, pred_log))
print("Confusion:\n", confusion_matrix(
    y_test, pred_log, labels=["<30", ">30", "NO"]))

Fitting 5 folds for each of 10 candidates, totalling 50 fits


KeyboardInterrupt: 

## 2) Random Forest

In [ ]:
pipe_rf = Pipeline(steps=[
    ("pre", pre),
    ("clf", RandomForestClassifier(
        n_estimators=400, random_state=42, n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

param_rf = {
    "clf__max_depth": [None, 8, 12, 20],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__max_features": ["sqrt", "log2", None],
}

search_rf = RandomizedSearchCV(
    pipe_rf, param_distributions=param_rf, n_iter=12,
    scoring=scorer, cv=cv, n_jobs=-1, random_state=42, verbose=1
)
search_rf.fit(X_train, y_train)
best_rf = search_rf.best_estimator_
pred_rf = best_rf.predict(X_test)

print("RF best:", search_rf.best_params_)
print(classification_report(y_test, pred_rf))
print("Confusion:\n", confusion_matrix(
    y_test, pred_rf, labels=["<30", ">30", "NO"]))

## 3) HistGradientBoosting (sklearn)

In [ ]:
pipe_hgb = Pipeline(steps=[
    ("pre", pre),
    ("clf", HistGradientBoostingClassifier(random_state=42))
])

param_hgb = {
    "clf__learning_rate": [0.03, 0.05, 0.1],
    "clf__max_depth": [3, 5, None],
    "clf__max_iter": [200, 400],
    "clf__min_samples_leaf": [20, 50, 100],
}

search_hgb = RandomizedSearchCV(
    pipe_hgb, param_distributions=param_hgb, n_iter=12,
    scoring=scorer, cv=cv, n_jobs=-1, random_state=42, verbose=1
)
search_hgb.fit(X_train, y_train)
best_hgb = search_hgb.best_estimator_
pred_hgb = best_hgb.predict(X_test)

print("HGB best:", search_hgb.best_params_)
print(classification_report(y_test, pred_hgb))
print("Confusion:\n", confusion_matrix(
    y_test, pred_hgb, labels=["<30", ">30", "NO"]))

## 4) Comparatif final
TODO: Construire un tableau Macro F1 + Recall "<30" pour chaque modèle et choisir le modèle final.

In [ ]:
def model_summary(y_true, y_pred, name):
    # Macro F1
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    # Recall de la classe "<30"
    labels = ["<30", ">30", "NO"]
    pr, rc, f1s, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)
    recall_lt30 = rc[0]
    return {"model": name, "macro_f1": macro_f1, "recall_<30": recall_lt30}


summary = pd.DataFrame([
    model_summary(y_test, pred_log, "LogReg"),
    model_summary(y_test, pred_rf, "RandomForest"),
    model_summary(y_test, pred_hgb, "HistGradientBoosting"),
]).sort_values("macro_f1", ascending=False)

summary